In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
# read the db_fields
rich = pd.read_csv('', dtype={'id_number' : str,
                                             'cellphone_number' : str})

In [ ]:
pathToFile = ""

In [ ]:
# general support service sheet
df1 = (
    pd.read_excel(
        io=pathToFile,
        sheet_name=0,
        skiprows=2,
        usecols=[0, 1, 3, 4, 5],
        names=['umuzi_email', 'cohort_name', 'service_used', 'service_name', 'date_service_accessed']
    )
    .assign(
        date_service_accessed = lambda df: pd.to_datetime(df['date_service_accessed']).dt.date,
        umuzi_email = lambda df: df['umuzi_email'].str.strip()
    )
)
df1.head()

In [ ]:
# connection support service sheet

df2 = (
    pd.read_excel(
        io=pathToFile,
        sheet_name=1,
        skiprows=2,
        usecols=[2, 3, 5, 6, 7],
        names=['umuzi_email', 'cohort_name', 'service_used', 'service_name', 'date_service_accessed']
    )
    .assign(
        umuzi_email = lambda df: df['umuzi_email'].str.split(","),
        date_service_accessed = lambda df: pd.to_datetime(df['date_service_accessed']).dt.date
    )
    .explode('umuzi_email', ignore_index=True)
    .assign(
        umuzi_email = lambda df: df['umuzi_email'].str.strip()
    )
)
df2.head()

In [ ]:
data = pd.concat([df1, df2], ignore_index=True)
data.head()

In [ ]:
data.columns

In [ ]:
# Identify emails that don't match on 'umuzi_email' but do match on 'email' in the rich dataset
# by finding the difference between unmatchable emails by umuzi_email and unmatchable by email
a = set(set(data['umuzi_email'])).difference(set(rich['email']))
b = set(set(data['umuzi_email'])).difference(set(rich['umuzi_email']))

b - a

In [ ]:
set(data['umuzi_email']).difference(set(rich['umuzi_email'])).__len__()

In [ ]:
[entry.strip() for entry in b]

> the above emails either can't be found or the holders are not of South African decent

In [ ]:
data.drop(
    labels = data.loc[data['umuzi_email'].isin(b)].index,
    axis=0, inplace=True
)

In [ ]:
premerge_rows = data.shape[0]

In [ ]:
data = data.merge(
    rich.drop(columns=['email']), 
    on='umuzi_email', 
    how='left')

In [ ]:
assert premerge_rows == data.shape[0], "Possible duplicates introduced during merging due to non-uniqueness of merge keys!"

In [ ]:
data.head()

In [ ]:
data['date_service_accessed'] = pd.to_datetime(data['date_service_accessed']).dt.date

In [ ]:
data['age_service_accessed'] = (
    pd.to_datetime(data['date_service_accessed'])- pd.to_datetime(data['date_of_birth'])
).dt.days // 365

# add age bins
data['age_range'] = pd.cut(
    x=data['age_service_accessed'],
    bins=[-np.inf, 17, 25, 35, np.inf],
    labels=['17 and below', '18-25', '26-35', '36+']
)

In [ ]:
data['month_of_service_accessed'] = pd.to_datetime(data['date_service_accessed']).dt.month_name()

data['month_of_service_accessed'] = data['month_of_service_accessed'] + ' 2025' # type: ignore

In [ ]:
data = data[[
    'learner_id', 'application_id', 'umuzi_email',
    'first_name', 'last_name', 'cohort_name', 'cellphone_number',
    'id_number', 'gender', 'date_of_birth', 'age_service_accessed', 'age_range', 'race',
    'residential_area_type', 'has_disability_or_differently_abled',
    'application_date', 'nearest_metro', 'province', 'date_service_accessed',
    'month_of_service_accessed', 'service_used', 'service_name'
]]

In [ ]:
# export to ind7 data

# data.to_csv('../Sink Datasets/indicator_7_sap_support_services.csv', index=False)

In [ ]:
# append to existing ind7 data
# data.to_csv('../Sink Datasets/indicator_7_data.csv', index=False, mode='a', header=False)